In [1]:
# !pip install pandas numpy seaborn matplotlib

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [6]:
df = pd.read_csv("../../datasets/old-RNNs/treated bbr esmond data ap-ba 07-03-2023_longest_interval.csv")

print(df.head(10))

new_df = pd.DataFrame()

new_df['timestamp'] = pd.to_datetime(df['Timestamp'], format="%d-%m-%y %H:%M:%S", errors='coerce')
new_df = new_df.sort_values('timestamp').reset_index(drop=True)

new_df['throughput'] = df['Throughput']

new_df.head(10)

   Unnamed: 0          Timestamp   Throughput
0           0  04-01-23 00:00:00          NaN
1           1  04-01-23 06:00:00          NaN
2           2  04-01-23 12:00:00          NaN
3           3  04-01-23 18:00:00  849059002.5
4           4  05-01-23 00:00:00  912053206.0
5           5  05-01-23 06:00:00  912420459.0
6           6  05-01-23 12:00:00  922591115.0
7           7  05-01-23 18:00:00  921072349.0
8           8  06-01-23 00:00:00  920049754.5
9           9  06-01-23 06:00:00  922798939.0


,timestamp,throughput
0,2023-01-04 00:00:00,NaN
1,2023-01-04 06:00:00,NaN
2,2023-01-04 12:00:00,NaN
3,2023-01-04 18:00:00,849059002.5
4,2023-01-05 00:00:00,912053206.0
5,2023-01-05 06:00:00,912420459.0
6,2023-01-05 12:00:00,922591115.0
7,2023-01-05 18:00:00,921072349.0
8,2023-01-06 00:00:00,920049754.5
9,2023-01-06 06:00:00,922798939.0


In [7]:
df_feat = new_df.copy()

df_feat['hour'] = df_feat['timestamp'].dt.hour
df_feat['day_of_week'] = df_feat['timestamp'].dt.dayofweek

# Adjust based on global week start
# df_feat['relative_day'] = ((df_feat['day_of_week'] - WEEK_START_DAY) % 7)
# df_feat['is_weekend'] = df_feat['relative_day'] >= 5

# Continuous time since beginning (for drift learning)
df_feat['time_since_start'] = (df_feat['timestamp'] - df_feat['timestamp'].min()).dt.total_seconds() / 3600.0

# Encode cyclical time features
df_feat['hour_sin'] = np.sin(2 * np.pi * df_feat['hour'] / 4)
df_feat['hour_cos'] = np.cos(2 * np.pi * df_feat['hour'] / 4)

df_feat['is_real'] = ~df_feat['throughput'].isna()

df_feat['time_since_last_real'] = np.nan

last_real_time = None

for i, row in df_feat.iterrows():
    current_time = row['timestamp']
    if row['is_real']:
        df_feat.at[i, 'time_since_last_real'] = 0.0
        last_real_time = current_time
    else:
        if last_real_time is not None:
            delta = (current_time - last_real_time).total_seconds() / 3600.0  # em horas
            df_feat.at[i, 'time_since_last_real'] = delta
        else:
            df_feat.at[i, 'time_since_last_real'] = np.nan  # Sem referência anterior

df_feat.tail(20)

,timestamp,throughput,hour,day_of_week,time_since_start,hour_sin,hour_cos,is_real,time_since_last_real
704,2023-06-29 00:00:00,871497324.0,0,3,4224.0,0.000000e+00,1.0,True,0.0
705,2023-06-29 06:00:00,899317861.0,6,3,4230.0,3.673940e-16,-1.0,True,0.0
706,2023-06-29 12:00:00,NaN,12,3,4236.0,-7.347881e-16,1.0,False,6.0
707,2023-06-29 18:00:00,NaN,18,3,4242.0,1.102182e-15,-1.0,False,12.0
708,2023-06-30 00:00:00,NaN,0,4,4248.0,0.000000e+00,1.0,False,18.0
709,2023-06-30 06:00:00,NaN,6,4,4254.0,3.673940e-16,-1.0,False,24.0
710,2023-06-30 12:00:00,NaN,12,4,4260.0,-7.347881e-16,1.0,False,30.0
711,2023-06-30 18:00:00,899159000.0,18,4,4266.0,1.102182e-15,-1.0,True,0.0
712,2023-07-01 00:00:00,814431015.0,0,5,4272.0,0.000000e+00,1.0,True,0.0
713,2023-07-01 06:00:00,920632145.0,6,5,4278.0,3.673940e-16,-1.0,True,0.0


In [4]:
# TO-DO: This part needs to be united with filling missing data
# ROLLING_WINDOW = 4  # 4 intervals = 1 day (6h * 4)

# df_feat['throughput_rolling_mean'] = df_feat['Vazao'].rolling(window=ROLLING_WINDOW, min_periods=1).mean()
# df_feat['throughput_rolling_std'] = df_feat['Vazao'].rolling(window=ROLLING_WINDOW, min_periods=1).std()